# 第 8 讲实验：ZeRO / FSDP 内存账本与 Activation Checkpointing

本实验先建立显式的模型状态内存账本，再用可选的单 GPU probe 测量 activation checkpointing 对一个小型 MLP 的影响。内存公式是教学估算；单卡 probe 不是 FSDP/NCCL 多 GPU benchmark。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_08").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_08"))
from sharding_lab import (
    STAGE_NAMES, activation_checkpoint_tradeoff, bytes_to_gib,
    cuda_checkpointing_probe, estimate_zero_stage_memory, partition_sizes,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
else:
    print("CPU fallback: formula lab works; CUDA memory probe will be skipped.")

## 实验 1：ZeRO stage 的模型状态账本

参数量 P=1B；权重 FP16/BF16（2 bytes），梯度 FP32（4 bytes），Adam moments 按每参数 8 bytes 估算。这个模型不自动包含 activation、allocator fragmentation 或所有通信 workspace。

In [ ]:
P = 1_000_000_000
WORLD_SIZE = 4
WEIGHT_BYTES, GRAD_BYTES, OPT_BYTES = 2, 4, 8

stage_results = [
    estimate_zero_stage_memory(
        P, world_size=WORLD_SIZE, stage=stage,
        weight_bytes_per_param=WEIGHT_BYTES,
        gradient_bytes_per_param=GRAD_BYTES,
        optimizer_bytes_per_param=OPT_BYTES,
        gathered_parameter_count=50_000_000,
    )
    for stage in range(4)
]
for item in stage_results:
    print(
        f"{item['stage_name']}: parameters={bytes_to_gib(item['parameter_bytes']):.2f} GiB, "
        f"gradients={bytes_to_gib(item['gradient_bytes']):.2f} GiB, "
        f"optimizer={bytes_to_gib(item['optimizer_state_bytes']):.2f} GiB, "
        f"persistent={bytes_to_gib(item['persistent_state_bytes']):.2f} GiB, "
        f"transient gather={bytes_to_gib(item['transient_gather_bytes']):.2f} GiB"
    )
assert [x["persistent_state_bytes"] for x in stage_results] == sorted(
    [x["persistent_state_bytes"] for x in stage_results], reverse=True
)
print("Balanced shard sizes for 10 items over 4 ranks:", partition_sizes(10, 4))

## 实验 2：状态分片不是总峰值显存

下面只画 persistent model state。ZeRO-3/FSDP 在模块计算前可能临时 all-gather 当前 wrap unit 的参数；activation、communication buffers 与 allocator overhead 也会影响峰值。

In [ ]:
labels = ["DDP / Z0", "ZeRO-1", "ZeRO-2", "ZeRO-3 / FSDP"]
param_gib = [bytes_to_gib(x["parameter_bytes"]) for x in stage_results]
grad_gib = [bytes_to_gib(x["gradient_bytes"]) for x in stage_results]
opt_gib = [bytes_to_gib(x["optimizer_state_bytes"]) for x in stage_results]
plt.figure(figsize=(8, 4.5))
plt.bar(labels, param_gib, label="parameters")
plt.bar(labels, grad_gib, bottom=param_gib, label="gradients")
bottom2 = [p + g for p, g in zip(param_gib, grad_gib)]
plt.bar(labels, opt_gib, bottom=bottom2, label="optimizer states")
plt.ylabel("Persistent model state (GiB, base 1024)")
plt.title("Illustrative per-rank state memory, world size = 4")
plt.xticks(rotation=12)
plt.legend()
plt.tight_layout()
plt.show()

# Example of explicit peak components for one ZeRO-3/FSDP wrap unit.
peak_example = estimate_zero_stage_memory(
    P, world_size=WORLD_SIZE, stage=3,
    weight_bytes_per_param=WEIGHT_BYTES, gradient_bytes_per_param=GRAD_BYTES,
    optimizer_bytes_per_param=OPT_BYTES, gathered_parameter_count=50_000_000,
    activation_bytes=8 * 1024**3, communication_buffer_bytes=1 * 1024**3,
)
print("Illustrative ZeRO-3 peak with 8 GiB activations + 1 GiB comm buffer:",
      f"{bytes_to_gib(peak_example['estimated_peak_bytes']):.2f} GiB")
print("This is an input-driven estimate, not measured allocator peak.")

## 实验 3：Activation checkpointing 的显存/计算交换

Checkpointing 在反向传播时重算部分前向，通常减少需要保存的中间激活，但会增加计算。它不直接减少参数或 Adam optimizer state。

In [ ]:
tradeoff = activation_checkpoint_tradeoff(
    saved_activation_bytes=2 * 1024**3,
    recompute_flops=80_000_000_000,
)
print("Hypothetical checkpointing tradeoff:")
print("saved activations:", bytes_to_gib(tradeoff["saved_activation_bytes"]), "GiB")
print("extra recompute:", f"{tradeoff['recompute_flops']:,} FLOPs")
print("These inputs are illustrative, not measured model values.")

## 实验 4：可选 CUDA 单卡测量

若 CUDA 可用，将同一小型 MLP 在 checkpointing 关闭/开启时分别 warm-up，再使用 CUDA events 测量 steady-state step latency，并记录 peak allocated memory。它不是 FSDP benchmark，也不代表所有模型都会节省相同比例的显存。

In [ ]:
if torch.cuda.is_available():
    probe = cuda_checkpointing_probe(
        batch_size=2, sequence_length=256, hidden_dim=256,
        num_layers=3, expansion=4, warmup=2, iterations=5, seed=8,
    )
    print("metadata:", probe["metadata"])
    for key in ("checkpointing_off", "checkpointing_on"):
        item = probe[key]
        print(
            key,
            f"mean step={item['mean_step_ms']:.3f} ms",
            f"peak allocated={bytes_to_gib(item['peak_allocated_bytes']):.3f} GiB",
            f"peak above baseline={bytes_to_gib(item['peak_above_baseline_bytes']):.3f} GiB",
            f"final loss={item['final_loss']:.6f}",
        )
    keys = ["checkpointing_off", "checkpointing_on"]
    plt.figure(figsize=(5.5, 3.5))
    plt.bar(["off", "on"], [bytes_to_gib(probe[k]["peak_allocated_bytes"]) for k in keys])
    plt.ylabel("Peak allocated memory (GiB)")
    plt.title("Single-GPU toy MLP")
    plt.tight_layout()
    plt.show()
else:
    print("SKIP: CUDA is unavailable in this kernel; no GPU memory/latency result is claimed.")

## 复盘

1. ZeRO-1/2/3 分别切分哪些模型状态？
2. 为什么 ZeRO-3/FSDP 的 persistent state 不等于总峰值显存？
3. 为什么 checkpointing 不会直接减少参数和 optimizer states？
4. 单 GPU checkpointing probe 能证明什么，不能证明什么？
5. 若要验收真实 FSDP，需要补充哪些多进程、多卡、数值和通信测量？